# **LIMPIEZA DE DATOS**

*Fase 2 de metodología OSEMN: SCRUB (Limpieza)*

Data utilizada:



1.   ENSANUT 2018: Contiene los datos de 43.311 hogares donde se investigó la ENSANUT, encuesta que recopila información sobre salud sexual y reproductiva, salud en la niñez, estado nutricional,  acceso a programas de primera infancia, además de datos sobre acceso a los servicios de salud.
2.   ENDI: La base de datos de Salud niñez de la Encuesta Nacional sobre Desnutrición Infantil está compuesta de 21.333 registros, que corresponde a información de madres de niñas/os menores de 5 años investigadas en el año I de la Encuesta, cuenta con 412 variables. Para la recolección de la información se utiliza el formulario 2 (MEF), el mismo que fué consensuado en la Comisión Especial de Estadísticas de Salud.



***Etapa 1: Calidad del dataset***

Primero se revisa la calidad porque no conviene construir análisis sobre datos malos. Un dataset bien revisado produce resultados más fiables y decisiones mejor fundamentadas. Para esto realizaremos el proceso EDA, explorando las caracteristicas de la estructura del dataset para poder limpiarlo correctamente, porque no puedes limpiar algo que no conoces.

Lo que vamos a hacer no será el EDA analítico, sino un EDA técnico o auditoría del dataset, que es la fase que normalmente se realiza antes del Scrub. Así no tomaremos decisiones de limpieza sin conocer los datos.

Este notebook responderá preguntas como:

¿Cuántos registros y columnas tiene cada dataset?
¿Qué tipo de datos tiene cada variable?
¿Cuántos valores nulos existen?
¿Hay registros duplicados?
¿Cuáles son las variables con mayor cardinalidad?
¿Qué variables parecen ser identificadores?
¿Qué categorías contiene cada variable?
¿Existen valores atípicos evidentes en variables numéricas?
¿Qué variables serán útiles para el modelo?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from pandas_ods_reader import read_ods

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10,6)

In [ ]:
RUTA_ENSANUT = "/content/6_BDD_ENS2018_f2_salud_ninez.csv"
RUTA_ENDI = "/content/BDD_ENDI_R1_f2_salud_ninez.csv"

DICCIONARIO_ENSANUT = "/content/Diccionario de Datos ENSANUT 2018.ods"
DICCIONARIO_ENDI = "/content/Diccionario de BDD_ENDI_R1_salud_niñez.ods"

In [ ]:
ensanut = pd.read_csv(
    RUTA_ENSANUT,
    encoding="latin1",
    low_memory=False
)

In [ ]:
endi = pd.read_csv(
    RUTA_ENDI,
    encoding="latin1",
    sep=None,
    engine='python'
)

In [ ]:
print("ENSANUT:",ensanut.shape)
print("ENDI:",endi.shape)

ENSANUT: (20510, 350)
ENDI: (21333, 412)


In [ ]:
# ==========================================================
# FUNCIÓN PARA CORREGIR CODIFICACIÓN UTF8/LATIN1
# ==========================================================

def corregir_codificacion(texto):
    if pd.isna(texto):
        return texto
    if not isinstance(texto, str):
        return texto
    try:
        return texto.encode("latin1").decode("utf8")
    except:
        return texto

In [ ]:
columnas = ensanut.select_dtypes(include="object").columns
for c in columnas:
    ensanut[c] = ensanut[c].apply(corregir_codificacion)
print("Codificación corregida en ENSANUT")


columnas = endi.select_dtypes(include="object").columns
for c in columnas:
    endi[c] = endi[c].apply(corregir_codificacion)
print("Codificación corregida en ENDI")

Codificación corregida en ENSANUT
Codificación corregida en ENDI


In [ ]:
ensanut.columns = [
    corregir_codificacion(c)
    for c in ensanut.columns
]

endi.columns = [
    corregir_codificacion(c)
    for c in endi.columns
]

In [ ]:
def limpiar_texto(texto):
    if pd.isna(texto):
        return texto
    if not isinstance(texto,str):
        return texto
    texto = texto.strip()
    texto = re.sub(r"\s+"," ",texto)
    return texto

In [ ]:
for c in ensanut.select_dtypes(include="object"):
    ensanut[c] = ensanut[c].apply(limpiar_texto)

for c in endi.select_dtypes(include="object"):
    endi[c] = endi[c].apply(limpiar_texto)

In [ ]:
errores = ["Ã","Â","�"]
def revisar_codificacion(df):
    columnas_error=[]
    for c in df.select_dtypes(include="object"):
        texto=df[c].astype(str)
        if texto.str.contains("|".join(errores),regex=True).any():
            columnas_error.append(c)
    return columnas_error

In [ ]:
dic_ensanut = read_ods(
    DICCIONARIO_ENSANUT,
    1
)

dic_endi = read_ods(
    DICCIONARIO_ENDI,
    1
)

In [ ]:
display(dic_ensanut.head())
display(dic_endi.head())

,Nombre del campo,Descripción del campo
0,area,Área
1,prov,Provincia
2,upm,Indentificador de upm
3,id_viv,Indentificador de vivienda
4,id_hogar,Indentificador del hogar


,Nombre del campo,Descripción del campo
0,id_upm,Identificador de upm
1,id_viv,Identificador de vivienda
2,id_hogar,Identificador del hogar
3,id_mef,Identificador de MEF
4,id_per,Identificador del hijo/a (usar para emparejar con f1_personas)


In [ ]:
diccionario_ensanut = dict(
zip(
dic_ensanut["Nombre del campo"],
dic_ensanut["Descripción del campo"]
)
)

diccionario_endi = dict(
zip(
dic_endi["Nombre del campo"],
dic_endi["Descripción del campo"]
)
)

In [ ]:
def renombrar_columnas(df, diccionario):
    nuevas_columnas = []
    for columna in df.columns:
        if columna in diccionario:
            nuevas_columnas.append(diccionario[columna])
        else:
            nuevas_columnas.append(columna)
    df.columns = nuevas_columnas
    return df

In [ ]:
ensanut_limpio = renombrar_columnas(
    ensanut,
    diccionario_ensanut
)

endi_limpio = renombrar_columnas(
    endi,
    diccionario_endi
)

In [ ]:
#====================================================
# INFORMACIÓN GENERAL DEL DATASET
#====================================================

def informacion_general(df, nombre):

    print("="*90)
    print(f"DATASET: {nombre}")
    print("="*90)

    print(f"Número de filas    : {df.shape[0]:,}")
    print(f"Número de columnas : {df.shape[1]:,}")

    print("\nPrimeras 10 columnas:")

    for c in df.columns[:10]:
        print("-", c)

    print("\nÚltimas 10 columnas:")

    for c in df.columns[-10:]:
        print("-", c)

In [ ]:
print("="*60)
informacion_general(ensanut,"ENSANUT")

print("="*60)
informacion_general(endi,"ENDI")

DATASET: ENSANUT
Número de filas    : 20,510
Número de columnas : 350

Primeras 10 columnas:
- Área
- Provincia
- Indentificador de upm
- Indentificador de vivienda
- Indentificador del hogar
- Indentificador de la madre
- Indentificador de la hijo
- Indentificador del hijo según el orden
- Orden del hijo
- Cód. Persona

Últimas 10 columnas:
- Fecha de visita - mes
- Fecha de visita - día
- Región de la mef
- Identificación étnica de la mef
- Edad en meses
- Grupo edad en meses
- Nivel de instrucción de la mef
- Desnutrición crónica
- Factor de expansión
- Estratos
DATASET: ENDI
Número de filas    : 21,333
Número de columnas : 412

Primeras 10 columnas:
- ﻿id_upm
- Identificador de vivienda
- Identificador del hogar
- Identificador de MEF
- Identificador del hijo/a (usar para emparejar con f1_personas)
- Identificador del hijo/a (usar para emparejar con f2_salud_ninez)
- Identificador del hijo/a según orden de registro de la sección 2
- Fecha de la entrevista - año
- Fecha de la entrev

In [ ]:
endi = endi.replace(".", pd.NA)

In [ ]:
#====================================================
# TIPO DE VARIABLES
#====================================================

def resumen_variables(df):
    resumen = pd.DataFrame({
        "Variable": df.columns,
        "Tipo": df.dtypes.astype(str),
        "Valores Únicos": df.nunique(),
        "Nulos": df.isna().sum(),
        "% Nulos": round(df.isna().mean()*100,2)
    })
    return resumen

In [ ]:
resumen_ensanut = resumen_variables(ensanut)
display(resumen_ensanut.head(20))
display(resumen_ensanut.tail(20))
print("="*60)
resumen_endi = resumen_variables(endi)
display(resumen_endi.head(20))
display(resumen_endi.tail(20))

,Variable,Tipo,Valores Únicos,Nulos,% Nulos
Área,Área,str,832,0,0.00
Provincia,Provincia,float64,25,830,4.05
Indentificador de upm,Indentificador de upm,float64,2574,830,4.05
Indentificador de vivienda,Indentificador de vivienda,float64,16556,830,4.05
Indentificador del hogar,Indentificador del hogar,float64,16592,830,4.05
Indentificador de la madre,Indentificador de la madre,float64,16918,830,4.05
Indentificador de la hijo,Indentificador de la hijo,float64,18060,830,4.05
Indentificador del hijo según el orden,Indentificador del hijo según el orden,object,19680,830,4.05
Orden del hijo,Orden del hijo,float64,3,830,4.05
Cód. Persona,Cód. Persona,float64,18,830,4.05


,Variable,Tipo,Valores Únicos,Nulos,% Nulos
487.A7 PENTAVALENTE 3 - Recibió en el:,487.A7 PENTAVALENTE 3 - Recibió en el:,str,12,1211,5.90
487.A8 IPV 1 - Recibió en el:,487.A8 IPV 1 - Recibió en el:,str,12,1208,5.89
487.A9 OPV 2 - Recibió en el:,487.A9 OPV 2 - Recibió en el:,str,12,1208,5.89
487.A10 OPV 3 - Recibió en el:,487.A10 OPV 3 - Recibió en el:,str,12,1211,5.90
487.A11 NEUMOCOCO CJ 1 - Recibió en el:,487.A11 NEUMOCOCO CJ 1 - Recibió en el:,str,12,1208,5.89
487.A12 NEUMOCOCO CJ 2 - Recibió en el:,487.A12 NEUMOCOCO CJ 2 - Recibió en el:,str,12,1209,5.89
487.A13 NEUMOCOCO CJ 3 - Recibió en el:,487.A13 NEUMOCOCO CJ 3 - Recibió en el:,str,12,1217,5.93
487.A14 SRP 1 - Recibió en el:,487.A14 SRP 1 - Recibió en el:,str,12,1222,5.96
487.A15 SRP 2 - Recibió en el:,487.A15 SRP 2 - Recibió en el:,str,12,1225,5.97
Fecha de visita -año,Fecha de visita -año,float64,2,1103,5.38


,Variable,Tipo,Valores Únicos,Nulos,% Nulos
﻿id_upm,﻿id_upm,int64,2829,0,0.00
Identificador de vivienda,Identificador de vivienda,int64,18407,0,0.00
Identificador del hogar,Identificador del hogar,int64,18443,0,0.00
Identificador de MEF,Identificador de MEF,int64,18955,0,0.00
Identificador del hijo/a (usar para emparejar con f1_personas),Identificador del hijo/a (usar para emparejar con f1_personas),int64,21333,0,0.00
Identificador del hijo/a (usar para emparejar con f2_salud_ninez),Identificador del hijo/a (usar para emparejar con f2_salud_ninez),int64,21333,0,0.00
Identificador del hijo/a según orden de registro de la sección 2,Identificador del hijo/a según orden de registro de la sección 2,object,21333,0,0.00
Fecha de la entrevista - año,Fecha de la entrevista - año,int64,2,0,0.00
Fecha de la entrevista - mes,Fecha de la entrevista - mes,int64,12,0,0.00
Fecha de la entrevista - día,Fecha de la entrevista - día,int64,31,0,0.00


,Variable,Tipo,Valores Únicos,Nulos,% Nulos
Año que aplicaron la vacuna Influenza 3,Año que aplicaron la vacuna Influenza 3,str,6,19265,90.31
"¿Según madre, tiene dosis? Influenza 3","¿Según madre, tiene dosis? Influenza 3",str,3,2210,10.36
Asiste a CDI o Educación inicial,Asiste a CDI o Educación inicial,str,2,142,0.67
Asistió a CDI o Educación inicial,Asistió a CDI o Educación inicial,str,2,6471,30.33
Porque no asiste o participa de un CDI o E.I.,Porque no asiste o participa de un CDI o E.I.,str,8,7318,34.30
CDI o Educación inicial al que asite o asistió,CDI o Educación inicial al que asite o asistió,str,5,14157,66.36
Años de asistencia,Años de asistencia,str,5,14157,66.36
Meses de asistencia,Meses de asistencia,str,12,14157,66.36
Días a la semana,Días a la semana,str,6,14157,66.36
Horas al día,Horas al día,str,12,14157,66.36


In [ ]:
#====================================================
# RESUMEN DE TIPOS DE DATOS
#====================================================

print("ENSANUT")
display(ensanut.dtypes.value_counts())
print()
print("ENDI")
display(endi.dtypes.value_counts())

ENSANUT


str        184
float64    165
object       1
Name: count, dtype: int64


ENDI


str        393
int64       17
object       1
float64      1
Name: count, dtype: int64

In [ ]:
#====================================================
# COLUMNAS DUPLICADAS
#====================================================

duplicadas = ensanut.columns[ensanut.columns.duplicated()]
print("ENSANUT")
print(list(duplicadas))
print(len(duplicadas))
print()
duplicadas = endi.columns[endi.columns.duplicated()]
print("ENDI")
print(list(duplicadas))
print(len(duplicadas))

ENSANUT
['449.  ¿Qué tiempo después de nacido (…), le llevó al control médico por primera', '449.  ¿Qué tiempo después de nacido (…), le llevó al control médico por primera']
2

ENDI
['Con qué frecuencia', 'Con qué frecuencia', 'Cuántos', 'Cuántos', 'Cuántos', 'Cuántos', 'Cuántos', 'Cuántos', 'Cuántos', 'Cuántos', 'Cuántas (último/a nacido/a vivo/a)', 'Dónde la recibió principalmente (último/a nacido/a vivo/a)', 'Cuántas (último/a nacido/a vivo/a)', 'Dónde la recibió principalmente (último/a nacido/a vivo/a)', 'Cuántas (último/a nacido/a vivo/a)', 'Dónde la recibió principalmente (último/a nacido/a vivo/a)', 'Cuántas (último/a nacido/a vivo/a)', 'Dónde la recibió principalmente (último/a nacido/a vivo/a)', 'Cuántas (último/a nacido/a vivo/a)', 'Dónde la recibió principalmente (último/a nacido/a vivo/a)', 'Cuántas (último/a nacido/a vivo/a)', 'Cuántas (último/a nacido/a vivo/a)', 'Dónde la recibió principalmente (último/a nacido/a vivo/a)', 'Cuántas (último/a nacido/a vivo/a)', 'Dónde l

In [ ]:
print("ENSANUT")
print(ensanut.duplicated().sum())
print()
print("ENDI")
print(endi.duplicated().sum())

ENSANUT
0

ENDI
0


In [ ]:
#====================================================
# VARIABLES CONSTANTES
#====================================================

def variables_constantes(df):
    constantes = []
    for columna in df.columns:
        try:
            # Si existen columnas duplicadas con el mismo nombre,
            # se toma únicamente la primera
            serie = df[columna]
            if isinstance(serie, pd.DataFrame):
                serie = serie.iloc[:,0]
            if serie.nunique(dropna=False) == 1:
                constantes.append(columna)
        except Exception as e:
            print(f"Error en {columna}: {e}")
    return constantes

In [ ]:
constantesE=variables_constantes(ensanut)
print("Variables constantes:")
len(constantesE)
constantesE[:30]
constantes=variables_constantes(endi)
print("Variables constantes:")
len(constantes)
constantes[:30]

Variables constantes:
Variables constantes:


[]

In [ ]:
#====================================================
# CARDINALIDAD
#====================================================

def cardinalidad(df):
    tabla = pd.DataFrame({
        "Variable":df.columns,
        "Valores Únicos":df.nunique()
    })

    return tabla.sort_values(
        "Valores Únicos",
        ascending=False
    )

In [ ]:
display(cardinalidad(ensanut).head(30))
display(cardinalidad(endi).head(30))

,Variable,Valores Únicos
Indentificador del hijo según el orden,Indentificador del hijo según el orden,19680
Indentificador de la hijo,Indentificador de la hijo,18060
Indentificador de la madre,Indentificador de la madre,16918
Indentificador del hogar,Indentificador del hogar,16592
Indentificador de vivienda,Indentificador de vivienda,16556
Factor de expansión,Factor de expansión,9905
Indentificador de upm,Indentificador de upm,2574
436.b gramos,436.b gramos,999
Área,Área,832
437.b Peso:,437.b Peso:,637


,Variable,Valores Únicos
Identificador del hijo/a (usar para emparejar con f2_salud_ninez),Identificador del hijo/a (usar para emparejar con f2_salud_ninez),21333
Identificador del hijo/a según orden de registro de la sección 2,Identificador del hijo/a según orden de registro de la sección 2,21333
Identificador del hijo/a (usar para emparejar con f1_personas),Identificador del hijo/a (usar para emparejar con f1_personas),21333
Identificador de MEF,Identificador de MEF,18955
Identificador del hogar,Identificador del hogar,18443
Identificador de vivienda,Identificador de vivienda,18407
﻿id_upm,﻿id_upm,2829
Factor de expansión,Factor de expansión,2784
Cantidad peso,Cantidad peso,1144
Peso al nacer gramos,Peso al nacer gramos,610


In [ ]:
#====================================================
# VALORES NULOS
#====================================================
def nulos(df):
    tabla = pd.DataFrame({
        "Variable":df.columns,
        "Nulos":df.isna().sum(),
        "Porcentaje":
        round(
            df.isna().mean()*100,
            2
        )
    })
    return tabla.sort_values(
        "Porcentaje",
        ascending=False
    )

In [ ]:
nulos_ensanut = nulos(ensanut)
display(nulos_ensanut.head(30))

nulos_endi = nulos(endi)
display(nulos_endi.head(30))

,Variable,Nulos,Porcentaje
451.a24 Control24 día,451.a24 Control24 día,20504,99.97
451.a24 Control24,451.a24 Control24,20504,99.97
451.a24 Control24 año,451.a24 Control24 año,20504,99.97
451.a24 Control24 mes,451.a24 Control24 mes,20504,99.97
451.a23 Control23 día,451.a23 Control23 día,20502,99.96
451.a23 Control23,451.a23 Control23,20502,99.96
451.a23 Control23 año,451.a23 Control23 año,20502,99.96
451.a23 Control23 mes,451.a23 Control23 mes,20502,99.96
451.a22 Control22 mes,451.a22 Control22 mes,20498,99.94
451.a22 Control22 año,451.a22 Control22 año,20498,99.94


,Variable,Nulos,Porcentaje
Inscripción de fallecimiento,Inscripción de fallecimiento,21238,99.55
Mes del control 12 (último/a nacido/a vivo/a),Mes del control 12 (último/a nacido/a vivo/a),20630,96.70
Talla del control 12 (último/a nacido/a vivo/a),Talla del control 12 (último/a nacido/a vivo/a),20630,96.70
Día del control 12 (último/a nacido/a vivo/a),Día del control 12 (último/a nacido/a vivo/a),20630,96.70
Edad en meses del control 12 (último/a nacido/a vivo/a),Edad en meses del control 12 (último/a nacido/a vivo/a),20630,96.70
Año del control 12 (último/a nacido/a vivo/a),Año del control 12 (último/a nacido/a vivo/a),20630,96.70
Peso del control 12 (último/a nacido/a vivo/a),Peso del control 12 (último/a nacido/a vivo/a),20630,96.70
Talla del control 11 (último/a nacido/a vivo/a),Talla del control 11 (último/a nacido/a vivo/a),20425,95.74
Peso del control 11 (último/a nacido/a vivo/a),Peso del control 11 (último/a nacido/a vivo/a),20425,95.74
Día del control 11 (último/a nacido/a vivo/a),Día del control 11 (último/a nacido/a vivo/a),20425,95.74


In [ ]:
print("ENSANUT")
ensanut.info(memory_usage="deep")

print("ENDI")
endi.info(memory_usage="deep")

ENSANUT
<class 'pandas.DataFrame'>
RangeIndex: 20510 entries, 0 to 20509
Columns: 350 entries, Área to Estratos
dtypes: float64(165), object(1), str(184)
memory usage: 74.7 MB
ENDI
<class 'pandas.DataFrame'>
RangeIndex: 21333 entries, 0 to 21332
Columns: 412 entries, ﻿id_upm to ¿Dónde se realiza el cuidado
dtypes: float64(1), int64(17), object(1), str(393)
memory usage: 83.3 MB


In [ ]:
numericas_ensanut = ensanut.select_dtypes(include=np.number).columns.tolist()
categoricas_ensanut = ensanut.select_dtypes(exclude=np.number).columns.tolist()
print("Variables numéricas:",len(numericas_ensanut))
print("Variables categóricas:",len(categoricas_ensanut))

numericas_endi = endi.select_dtypes(include=np.number).columns.tolist()
categoricas_endi = endi.select_dtypes(exclude=np.number).columns.tolist()
print("Variables numéricas:",len(numericas_endi))
print("Variables categóricas:",len(categoricas_endi))

Variables numéricas: 165
Variables categóricas: 185
Variables numéricas: 18
Variables categóricas: 394


In [ ]:
display(ensanut.describe(include="all").T)
print("="*60)
display(endi.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Área,20510,832,urbano,11912,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Provincia,19680.0,NaN,NaN,NaN,13.17749,8.609373,1.0,7.0,12.0,18.0,90.0
Indentificador de upm,19680.0,NaN,NaN,NaN,132183353434.221802,86044448179.86264,10150000201.0,71350000901.0,121050001301.0,180750001305.0,900451900501.0
Indentificador de vivienda,19680.0,NaN,NaN,NaN,13218335343431.914062,8604444817986.282227,1015000020102.0,7135000090117.75,12105000130116.5,18075000130507.25,90045190050116.0
Indentificador del hogar,19680.0,NaN,NaN,NaN,132183353434320.15625,86044448179863.109375,10150000201021.0,71350000901178.5,121050001301166.0,180750001305073.5,900451900501161.0
...,...,...,...,...,...,...,...,...,...,...,...
Grupo edad en meses,19679,8,48-59,4132,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Nivel de instrucción de la mef,19680,4,Educación Media/Bachillerato,8346,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Desnutrición crónica,18493.0,NaN,NaN,NaN,0.24658,0.431032,0.0,0.0,0.0,0.0,1.0
Factor de expansión,19680.0,NaN,NaN,NaN,71.54755,99.043053,2.862436,21.996113,40.537766,75.154808,1402.8585


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
﻿id_upm,21333.0,NaN,NaN,NaN,1253143573.579665,676027645.085451,101500032.0,706500026.0,1301530006.0,1801500224.0,2403520025.0
Identificador de vivienda,21333.0,NaN,NaN,NaN,125314357362.454132,67602764508.544144,10150003201.0,70650002603.0,130153000601.0,180150022401.0,240352002508.0
Identificador del hogar,21333.0,NaN,NaN,NaN,12531435736246.421875,6760276450854.423828,1015000320101.0,7065000260301.0,13015300060101.0,18015002240101.0,24035200250802.0
Identificador de MEF,21333.0,NaN,NaN,NaN,1253143573624644.25,676027645085438.75,101500032010102.0,706500026030102.0,1301530006010102.0,1801500224010101.0,2403520025080201.0
Identificador del hijo/a (usar para emparejar con f1_personas),21333.0,NaN,NaN,NaN,1253143573624647.5,676027645085438.75,101500032010104.0,706500026030106.0,1301530006010105.0,1801500224010103.0,2403520025080204.0
...,...,...,...,...,...,...,...,...,...,...,...
Hace cuántas semanas fue visitado,7027,4,0,5839,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Hace cuantos meses fue visitado,7027,26,0,3257,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Persona que recibe o recibió la atención CNH,7027,7,Madre,6535,NaN,NaN,NaN,NaN,NaN,NaN,NaN
¿Con quién permanece la mayor parte del tiempo?,21191,10,Madre,16573,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(ensanut.columns[:20])
display(ensanut.head())

print(endi.columns[:20])
display(endi.head())

Index(['Área', 'Provincia', 'Indentificador de upm',
       'Indentificador de vivienda', 'Indentificador del hogar',
       'Indentificador de la madre', 'Indentificador de la hijo',
       'Indentificador del hijo según el orden', 'Orden del hijo',
       'Cód. Persona', 'Sexo', '402.  ¿Está vivo (…)?',
       '403.  En la época en la que quedó embarazada de (…), quería Usted:',
       '404.1 ¿Cuánto tiempo más hubiera querido esperar para el embarazo de (…)?: mese',
       '404.2 ¿Cuánto tiempo más hubiera querido esperar para el embarazo de (…)?: años',
       '405.  ¿Quería su pareja:',
       '406.  ¿Tuvo algún control prenatal cuando estaba embarazada de (...)?',
       '407.a ¿Dónde se hizo el control con mayor frecuencia?',
       '408.  ¿Consumió algún micronutriente durante el embarazo como:',
       '409.a ¿Con qué frecuencia tomaba Los micronutrientes?'],
      dtype='str')


,Área,Provincia,Indentificador de upm,Indentificador de vivienda,Indentificador del hogar,Indentificador de la madre,Indentificador de la hijo,Indentificador del hijo según el orden,Orden del hijo,Cód. Persona,Sexo,402. ¿Está vivo (…)?,"403. En la época en la que quedó embarazada de (…), quería Usted:",404.1 ¿Cuánto tiempo más hubiera querido esperar para el embarazo de (…)?: mese,404.2 ¿Cuánto tiempo más hubiera querido esperar para el embarazo de (…)?: años,405. ¿Quería su pareja:,406. ¿Tuvo algún control prenatal cuando estaba embarazada de (...)?,407.a ¿Dónde se hizo el control con mayor frecuencia?,408. ¿Consumió algún micronutriente durante el embarazo como:,409.a ¿Con qué frecuencia tomaba Los micronutrientes?,410. ¿Durante el embarazo le pesaron?,411. ¿Durante el embarazo le midieron la barriga?,412.a ¿Durante el embarazo le hicieron un examen de VIH?,412.b Cuántos?,413. ¿Durante el embarazo ¿le tomaron la presión?,414. ¿Durante el embarazo ¿hicieron un examen de sangre?,415. ¿Durante el embarazo ¿hicieron un examen de orina?,416. ¿Durante el embarazo ¿hicieron un examen de sífilis?,417. ¿Durante el embarazo ¿le vacunaron contra el tétanos?,418. ¿Cuántas veces le vacunaron contra el tétanos?,"419.a ¿Durante el control del embarazo, recibió consejería o asesoría sobre: A","419.b Uso de micronutrientes (hierro, ácido fólico)?","419.c Signos de alarma del embarazo (sangrado vaginal, falta de movimiento de b",419.d Higiene en preparación de alimentos?,419.e Lavado de manos?,419.f Métodos anticonceptivos?,"419.g El apego inmediato, lactancia en la primera hora de vida y el corte oport",420. ¿Cuántas semanas de embarazo tenía cuando le hicieron el primer control?,"421. En total, ¿cuántos controles tuvo antes del parto?",422. Antes de este embarazo ¿Le vacunaron a usted contra el tétanos (para prote,423.a ¿En que lugar tuvo el parto de (..).?,424. ¿Qué persona ó profesional le tendió?,425. El parto de (...) fue:,"426. ¿El nacimiento de (...) fue a los 9 meses o antes de tiempo (prematuro, si",427. ¿Cuántas semanas antes ó después de lo esperado nació (...),428. ¿Le pesaron a (…) en el momento de nacer o en los primeros 7 días?,429. ¿Le Realizaron a (…) La prueba del Tamizaje Neonatal (pinchada del talón a,430. ¿Le bañaron a (…) antes de cumplir 24 horas de su nacimiento?,431. ¿Esperaron al menos un minuto para realizar el corte del cordón umbilical?,432. Tiene usted el carné de salud infantil o libreta integral de (…)?,433.a ENCUESTADOR/A OBSERVE SI EL CARNÉ REGISTRA PUNTOS EN LA CURVA DE CRECIMIE,433.b Cuántos?,434.a ENCUESTADOR/A OBSERVE SI EL CARNÉ REGISTRA: TALLA AL NACER,434.b cm,435.a ENCUESTADOR/A OBSERVE SI EL CARNÉ REGISTRA PERÍMETRO CEFÁLICO AL NACER,435.b cm,436.a ENCUESTADOR/A OBSERVE SI EL CARNÉ REGISTRA:PESO AL NACER,436.b gramos,437.a ¿Cuánto pesó (…)?,437.b Peso:,"438. ¿(…) Peso menos de 5.5 Libras, 2.5 Kilogramos o 2500 gramos?","439. En comparación con otros niños recién nacidos, ¿cómo considera que era el",440. ¿Tuvo usted algún control después del parto de (...)?,441.a ¿Cuánto tiempo después del parto de (...) tuvo su primer control post par,441.b Semanas - ¿Cuánto tiempo después del parto de (...) tuvo su primer contro,442. ¿Dónde tuvo el control de post parto?,441.c ¿Cuánto tiempo después del parto de (...) tuvo su primer control post par,443.a ¿A los cuántos meses después del nacimiento de (...) le volvió su regla ?,444. ¿A los cuántos meses después del nacimiento de (...) volvió a tener relaci,445. ¿(..) fue inscrito en el Registro Civil?,446. ¿Está vivo (…)?,447. ¿Fue inscrito el fallecimiento de (...) en el Registro Civil?,"448. ¿Después de que nació (...), le llevó para control médico?","449. ¿Qué tiempo después de nacido (…), le llevó al control médico por primera","449. ¿Qué tiempo después de nacido (…), le llevó al control médico por primera","449. ¿Qué tiempo después de nacido (…), le llevó al control médico por primera",450. ¿Porqué o para qué le llevó a (…):,451.a ¿De 0 a menos de 1 años ?,451

Index(['﻿id_upm', 'Identificador de vivienda', 'Identificador del hogar',
       'Identificador de MEF',
       'Identificador del hijo/a (usar para emparejar con f1_personas)',
       'Identificador del hijo/a (usar para emparejar con f2_salud_ninez)',
       'Identificador del hijo/a según orden de registro de la sección 2',
       'Fecha de la entrevista - año', 'Fecha de la entrevista - mes',
       'Fecha de la entrevista - día', 'Factor de expansión', 'Estrato',
       'Área', 'Región', 'Provincia',
       'Orden de registro del hijo/a según la sección 2',
       'Número de registros', 'Número de código del niño/a', 'Está vivo',
       'En la época en la que quedó embarazada quería usted'],
      dtype='str')


,﻿id_upm,Identificador de vivienda,Identificador del hogar,Identificador de MEF,Identificador del hijo/a (usar para emparejar con f1_personas),Identificador del hijo/a (usar para emparejar con f2_salud_ninez),Identificador del hijo/a según orden de registro de la sección 2,Fecha de la entrevista - año,Fecha de la entrevista - mes,Fecha de la entrevista - día,Factor de expansión,Estrato,Área,Región,Provincia,Orden de registro del hijo/a según la sección 2,Número de registros,Número de código del niño/a,Está vivo,En la época en la que quedó embarazada quería usted,Tuvo algún control prenatal cuando estaba embarazada,Cuántas semanas de embarazo tenía cuando le hicieron el primer control,Cuántos controles tuvo antes del parto,Dónde se hizo el control con mayor frecuencia,Consumió ácido fólico tres meses antes del embarazo,Consumió ácido fólico durante el embarazo,Con qué frecuencia,Consumió hierro durante el embarazo,Con qué frecuencia,Consumió algún otro tipo de micronutrientes durante el embarazo,Con qué frecuencia,Durante el embarazo le pesaron,Durante el embarazo le midieron la barriga,Durante el embarazo le tomaron la presión,Durante el embarazo le realizaron el examen de VIH antes de la semana 20,Durante el embarazo le realizaron el examen de VIH a partir de la semana 20,Durante el parto le realizaron el examen del VIH,Durante el embarazo le hicieron exámenes de sangre para medir la anemia antes de la semana 20,Cuántos,Durante el embarazo de le hicieron exámenes de sangre para medir la anemia a partir de la semana 20,Cuántos,Durante el embarazo le hicieron exámenes de orina antes de la semana 20,Cuántos,Durante el embarazo le hicieron exámenes de orina a partir de la semana 20,Cuántos,Durante el embarazo le realizaron exámenes de TORCHs antes de la semana 20,Cuántos,Durante el embarazo le realizaron exámenes de TORCHs a partir de la semana 20,Cuántos,Durante el embarazo le vacunaron contra el TÉTANOS y DIFTERIA,Cuántos,Durante el embarazo le colocaron la vacuna contra la INFLUENZA,Cuántos,Durante el embarazo le hicieron ECOS OBSTÉTRICOS,Cuántos,"Recibió alguna consejería, asesoría o charla durante el embarazo (último/a nacido/a vivo/a)",Recibió consejería sobre Lactancia materna exclusiva (último/a nacido/a vivo/a),Cuántas (último/a nacido/a vivo/a),Dónde la recibió principalmente (último/a nacido/a vivo/a),Recibió consejería sobre consumo de micronutrientes (último/a nacido/a vivo/a),Cuántas (último/a nacido/a vivo/a),Dónde la recibió principalmente (último/a nacido/a vivo/a),Recibió consejería sobre signos de alarma del embarazo (último/a nacido/a vivo/a),Cuántas (último/a nacido/a vivo/a),Dónde la recibió principalmente (último/a nacido/a vivo/a),Recibió consejería sobre higiene en preparación de alimentos (último/a nacido/a vivo/a),Cuántas (último/a nacido/a vivo/a),Dónde la recibió principalmente (último/a nacido/a vivo/a),Recibió consejería sobre lavado de manos (último/a nacido/a vivo/a),Cuántas (último/a nacido/a vivo/a),Dónde la recibió principalmente (último/a nacido/a vivo/a),Recibió consejería sobre Planificación Familiar (último/a nacido/a vivo/a),Cuántas (último/a nacido/a vivo/a),Dónde la recibió principalmente (último/a nacido/a vivo/a),"Recibió consejería sobre el apego inmediato, lactancia en la primera hora de vida (último/a nacido/a vivo/a)",Cuántas (último/a nacido/a vivo/a),Dónde la recibió principalmnete (último/a nacido/a vivo/a),Recibió consejería sobre alimentación saludable (último/a nacido/a vivo/a),Cuántas (último/a nacido/a vivo/a),Dónde la recibió principalmente (último/a nacido/a vivo/a),Recibió consejería sobre planificación del parto y transporte (último/a nacido/a vivo/a),Cuántas (último/a nacido/a vivo/a),Dónde la recibió principalmente (último/a nacido/a vivo/a),Recibió consejería sobre Agua segura (último/a nacido/a vivo/a),Cuántas (último/a nacido/a vivo/a),dónde la recibió principalmente (último/a nacido/a vivo/a),Lugar que tuvo el parto,Qué persona ó profesional le atendió,Tipo de par

In [ ]:
# ==========================================================
# EXPORTAR DATASETS CORREGIDOS
# ==========================================================

# Guardar ENSANUT
ensanut.to_csv(
    "ENSANUT_CORREGIDO.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardar ENDI
endi.to_csv(
    "ENDI_CORREGIDO.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Datasets exportados correctamente.")

Datasets exportados correctamente.
